## Essai Forecasting numéro 100000

In [1]:
import pandas as pd
import pickle
from typing import NamedTuple, Dict, List, Any
from tqdm import tqdm  # For progress bar during calculations

# Biogeme 
import biogeme.database as db
from biogeme.biogeme import BIOGEME
from biogeme.expressions import Expression, Variable
from biogeme.models import loglogit, logit, nested, lognested

# models pref
from models.logit_lmpc12_model3 import V_3, chosen_alternative
from models.logit_lmpc12_model4 import results_nested, nests

File biogeme.toml has been created


In [4]:
# Charger les données
df = pd.read_csv('models/lpmc12.dat', sep='\t')

# Scénario (i): Ajout de 1,5 £ au coût pour les utilisateurs de voiture
df_scenario1 = df.copy()
df_scenario1['cost_driving_fuel'] += 1.5

# Scénario (ii): Réduction de 20 % du coût des transports publics
df_scenario2 = df.copy()
df_scenario2['cost_transit'] *= 0.8

# Définir les tailles de population pour chaque segment
census = {
    'female_44_less': 2841376,
    'female_45_more': 1519948,
    'male_44_less': 2926408,
    'male_45_more': 1379198,
}

population_size = sum(census.values())

# Définir les filtres pour chaque segment
filters = {
    'female_44_less': (df['female'] == 1) & (df['age'] <= 44),
    'female_45_more': (df['female'] == 1) & (df['age'] > 44),
    'male_44_less': (df['female'] == 0) & (df['age'] <= 44),
    'male_45_more': (df['female'] == 0) & (df['age'] > 44),
}

# Count the sample size in each stratum
sample_segments = {
    segment_name: segment_rows.sum() for segment_name, segment_rows in filters.items()
}
print(f'Sample segments: {sample_segments}')

# Total sample size
total_sample = sum(sample_segments.values())
print(f'Sample size: {total_sample}')

weights = {
    segment_name: census[segment_name] * total_sample / (segment_size * population_size)
    for segment_name, segment_size in sample_segments.items()
}

# Ajouter les poids dans les datasets
for segment_name, segment_rows in filters.items():
    df.loc[segment_rows, 'weight'] = weights[segment_name]
    df_scenario1.loc[segment_rows, 'weight'] = weights[segment_name]
    df_scenario2.loc[segment_rows, 'weight'] = weights[segment_name]

# Définir les modèles et scénarios
model_market_share: dict[str, List] = {
    "Model 4": [V_3, results_nested],
    "Model 4 scenario 1": [V_3, results_nested],
    "Model 4 scenario 2": [V_3, results_nested],
}

# Créer des bases de données pour chaque scénario
database_original = db.Database("Original", df)
database_scenario1 = db.Database("Scenario 1", df_scenario1)
database_scenario2 = db.Database("Scenario 2", df_scenario2)

# Associer les bases de données à chaque modèle
databases = {
    "Model 4": database_original,
    "Model 4 scenario 1": database_scenario1,
    "Model 4 scenario 2": database_scenario2,
}

# Classe pour les résultats des parts de marché
class IndicatorTuple(NamedTuple):
    value: float
    lower: float
    upper: float

# Fonction pour calculer les parts de marché pondérées
def market_share(utilities: dict[int, Expression], results, database) -> dict[str, IndicatorTuple]:
    
    simulate = {
        'weight': Variable('weight'),
        'Prob. walk': nested(utilities, None, nests, 1),
        'Prob. cycle': nested(utilities, None, nests, 2),
        'Prob. PT': nested(utilities, None, nests, 3),
        'Prob. car': nested(utilities, None, nests, 4),
    }

    biosim = BIOGEME(database, simulate)
    simulated_values = biosim.simulate(results.get_beta_values())
    

    market_shares = {}
    
    for alt_name, prob_name in [
        ("Walking", "Prob. walk"),
        ("Cycling", "Prob. cycle"),
        ("Public transportation", "Prob. PT"),
        ("Car", "Prob. car"),
    ]:
        weighted_prob = simulated_values["weight"] * simulated_values[prob_name]
        mean_value = weighted_prob.sum() / simulated_values["weight"].sum()
        market_shares[alt_name] = IndicatorTuple(value=mean_value, lower=None, upper=None)

    return market_shares

# Fichier pickle pour sauvegarder ou charger les résultats
file_name = 'market_shares_forecastingV2.pickle'

try:
    # Lire les parts de marché depuis le fichier
    with open(file_name, 'rb') as f:
        all_market_shares = pickle.load(f)
        print(f'Market shares read from {file_name}')
except FileNotFoundError:
    # Calculer les parts de marché si le fichier n'existe pas
    all_market_shares: dict[str, Any] = {}
    for model, (utilities, results) in tqdm(model_market_share.items()):
        database = databases[model]
        all_market_shares[model] = market_share(utilities, results, database)
    
    # Sauvegarder les parts de marché
    with open(file_name, 'wb') as f:
        pickle.dump(all_market_shares, f)
    print(f'Market shares calculated and saved in {file_name}')


Sample segments: {'female_44_less': np.int64(1631), 'female_45_more': np.int64(1034), 'male_44_less': np.int64(1451), 'male_45_more': np.int64(884)}
Sample size: 5000
Market shares read from market_shares_forecastingV2.pickle


In [5]:
# Afficher les résultats
for model, shares in all_market_shares.items():
    print(f"\nMarket shares for {model}:")
    for mode, share in shares.items():
        print(f"  {mode}: {share.value:.2%}")


Market shares for Model 4:
  Walking: 16.93%
  Cycling: 2.92%
  Public transportation: 36.19%
  Car: 43.97%

Market shares for Model 4 scenario 1:
  Walking: 17.53%
  Cycling: 3.07%
  Public transportation: 40.63%
  Car: 38.77%

Market shares for Model 4 scenario 2:
  Walking: 16.89%
  Cycling: 2.90%
  Public transportation: 37.20%
  Car: 43.02%
